# pyg lean notebook

This is a notebook version of the pyg lean fork (https://github.com/zoomingmediaresearch/pyg-lean) of pyg (passable youtube grabber), created by Raemisch and Muehleder in the diggr project (https://github.com/diggr/pyg).
It is intended to be integrated into the ytma toolchain, used after step0 in order to harvest additional metadata and comments data from YouTube.
Some of the functionality is adjusted, now offering a resume for harvesting data in the video id list after a previous quota exceed interrupt.
In order to reset the harvesting, delete the harvest_state.db file created in the project folder.
At this point, only one project is supported at one time - multiproject support may be added later.

In [ ]:
import os
import re
import yaml
import sqlite3
import sys
from datetime import datetime
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError

class YTMAStep0bHarvester:
    def __init__(self, target_directory, config_filename="config.yml", db_filename="harvest_state.db", reply_mode="all"):
        """
        :param target_directory: Folder path containing project files
        :param reply_mode: Options are:
                           'all'  -> Fetch absolutely every deep comment reply thread (High quota cost).
                           '5'    -> Fetch only up to the first 5 bundled replies per thread.
                           'none' -> Fetch no comment replies at all.
        """
        self.target_dir = os.path.abspath(target_directory)
        self.config_path = os.path.join(self.target_dir, config_filename)
        self.db_path = os.path.join(self.target_dir, db_filename)
        
        # Validate selection flag early
        if reply_mode not in ["all", "5", "none"]:
            raise ValueError("reply_mode must be one of: 'all', '5', 'none'")
        self.reply_mode = reply_mode
        
        self.load_config()
        self.init_db()
        self.youtube = build("youtube", "v3", developerKey=self.api_key)
        
    def load_config(self):
        if not os.path.exists(self.config_path):
            raise FileNotFoundError(f"Configuration file missing at: {self.config_path}")
        with open(self.config_path, 'r') as f:
            cfg = yaml.safe_load(f)
        
        self.api_key = cfg['youtube']['api-key']
        self.project_name = cfg['project']['name']
        
        raw_out_dir = cfg['project'].get('dir', 'data')
        if os.path.isabs(raw_out_dir):
            self.output_dir = raw_out_dir
        else:
            self.output_dir = os.path.join(self.target_dir, raw_out_dir)
            
        os.makedirs(self.output_dir, exist_ok=True)

    def init_db(self):
        conn = sqlite3.connect(self.db_path)
        cursor = conn.cursor()
        cursor.execute('''
            CREATE TABLE IF NOT EXISTS completed_harvests (
                item_id TEXT PRIMARY KEY,
                item_type TEXT,
                harvested_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
            )
        ''')
        conn.commit()
        conn.close()

    def is_already_harvested(self, item_id):
        conn = sqlite3.connect(self.db_path)
        cursor = conn.cursor()
        cursor.execute("SELECT 1 FROM completed_harvests WHERE item_id = ?", (item_id,))
        res = cursor.fetchone()
        conn.close()
        return res is not None

    def mark_as_harvested(self, item_id, item_type):
        conn = sqlite3.connect(self.db_path)
        cursor = conn.cursor()
        try:
            cursor.execute("INSERT INTO completed_harvests (item_id, item_type) VALUES (?, ?)", (item_id, item_type))
            conn.commit()
        except sqlite3.IntegrityError:
            pass
        conn.close()

    def fetch_all_replies(self, parent_id):
        """Fetches all deep nested replies for a comment thread when there are > 5 replies."""
        replies = []
        next_page_token = None
        
        while True:
            try:
                request = self.youtube.comments().list(
                    part="snippet",
                    parentId=parent_id,
                    maxResults=100,
                    pageToken=next_page_token
                )
                response = request.execute()
                
                for item in response.get("items", []):
                    replies.append(item)
                    
                next_page_token = response.get("nextPageToken")
                if not next_page_token:
                    break
                    
            except HttpError as e:
                if e.resp.status == 403:
                    print(f"\n[CRITICAL] Quota exceeded while fetching deep replies for comment {parent_id}!")
                    sys.exit(0)
                print(f"      [WARNING] Could not fetch replies for comment {parent_id}: {e}")
                break
        return replies

    def fetch_comments(self, video_id):
        """Fetches top-level comments and applies filtering rules based on self.reply_mode flag."""
        threads = []
        next_page_token = None
        
        # Optimize endpoint API parts requested based on flags
        api_part = "snippet" if self.reply_mode == "none" else "snippet,replies"
        
        while True:
            try:
                request = self.youtube.commentThreads().list(
                    part=api_part,
                    videoId=video_id,
                    maxResults=100,
                    pageToken=next_page_token
                )
                response = request.execute()
                
                for thread in response.get("items", []):
                    
                    if self.reply_mode == "all":
                        snippet = thread.get("snippet", {})
                        total_reply_count = snippet.get("totalReplyCount", 0)
                        
                        if total_reply_count > 0:
                            has_replies_block = "replies" in thread
                            bundled_replies_count = len(thread["replies"]["comments"]) if has_replies_block else 0
                            
                            if total_reply_count > bundled_replies_count:
                                all_replies = self.fetch_all_replies(thread['id'])
                                if "replies" not in thread:
                                    thread["replies"] = {}
                                thread["replies"]["comments"] = all_replies
                                
                    elif self.reply_mode == "none":
                        # Explicitly ensure no automated partial response payload leaks through 
                        if "replies" in thread:
                            del thread["replies"]
                            
                    threads.append(thread)
                    
                next_page_token = response.get("nextPageToken")
                if not next_page_token:
                    break
                    
            except HttpError as e:
                if e.resp.status == 403:
                    print(f"\n[CRITICAL] YouTube API Quota exceeded while fetching comments for {video_id}! Exiting safely.")
                    sys.exit(0)
                print(f"      [WARNING] Could not fetch comments for video {video_id}: {e}")
                break
        return threads

    def process_group_item(self, item, item_type, group_dir):
        sanitized_id = re.sub(r'[\\/*?:"<>|]', '_', item['id'])
        file_filename = f"{item_type}_{sanitized_id}.yaml"
        file_path = os.path.join(group_dir, file_filename)
        
        with open(file_path, 'w', encoding='utf-8') as f:
            yaml.dump(item, f, default_flow_style=False)
            
        self.mark_as_harvested(item['id'], item_type)

    def harvest_ids(self, item_ids, item_type, group_dir):
        pending_ids = [idx for idx in item_ids if not self.is_already_harvested(idx)]
        
        if not pending_ids:
            print(f"   -> All {item_type} IDs in this group have already been successfully harvested.")
            return

        print(f"   -> Resuming fetch: {len(pending_ids)} / {len(item_ids)} items remaining.")

        for i in range(0, len(pending_ids), 50):
            batch = pending_ids[i:i+50]
            clean_ids = [b.split('/')[-1] for b in batch]
            id_string = ",".join(clean_ids)

            try:
                if item_type == "channels":
                    request = self.youtube.channels().list(part="snippet,contentDetails,statistics", id=id_string)
                    response = request.execute()
                    for item in response.get("items", []):
                        self.process_group_item(item, item_type, group_dir)
                else:
                    request = self.youtube.videos().list(part="snippet,contentDetails,statistics", id=id_string)
                    response = request.execute()
                    
                    for item in response.get("items", []):
                        print(f"   -> Fetching comments (Mode: '{self.reply_mode}') for video: {item['id']}")
                        item['comments'] = self.fetch_comments(item['id'])
                        
                        self.process_group_item(item, item_type, group_dir)
                    
            except HttpError as e:
                if e.resp.status == 403:
                    print("\n[CRITICAL] YouTube API Quota exceeded during metadata fetch! Exiting program safely.")
                    sys.exit(0)
                else:
                    print(f"API Error encountered: {e}")
                    return

    def process_harvest(self, yaml_filename, item_type):
        yaml_file_path = os.path.join(self.target_dir, yaml_filename)
        print(f"\nStarting harvest for '{yaml_filename}' in {self.target_dir}...")
        if not os.path.exists(yaml_file_path):
            print(f"Target list '{yaml_filename}' not found in {self.target_dir}. Skipping.")
            return

        with open(yaml_file_path, 'r') as f:
            groups = yaml.safe_load(f)
        
        for group_name, ids in groups.items():
            print(f"\nProcessing Group: {group_name} ({len(ids)} targets)")
            
            group_dir = os.path.join(self.output_dir, group_name)
            os.makedirs(group_dir, exist_ok=True)
            
            self.harvest_ids(ids, item_type, group_dir)
        
        print(f"\nSuccessfully finalized current run session execution loop.")

In [ ]:
# ==============================================================================
# EXECUTION INTERFACE 

PROJECT_FOLDER = "PROVIDE FOLDER NAME HERE"  # <-- Change this to your project folder name

# Change reply_mode variable to: "all", "5", or "none"

harvester = YTMAStep0bHarvester(target_directory=PROJECT_FOLDER,reply_mode="all")
harvester.process_harvest("videos.yml", "videos")
